# 04 · Entrenamiento de modelos

Primero revisa `03_aumento_datos.ipynb`. Ejecuta la comprobación y luego **solo la celda del modelo que quieras entrenar**; no uses Run All. Los clasificadores usan `data/splits`, preparado desde `USK-Coffee/train`.

Las épocas configuradas son un **máximo de 100**, no una obligación de llegar a 100. Slim-CNN y ResNet18 siamesa detienen el entrenamiento si `val_loss` no mejora de forma suficiente durante 15 épocas seguidas. YOLO aplica su parada temprana con la misma paciencia. En todos se conserva `best` para evaluación y `last` para reanudar; los historiales registran épocas, iteraciones, pérdidas, métricas y tiempo.

Este notebook ofrece YOLOv8-cls, Slim-CNN y ResNet18 siamesa. YOLOv8 de detección requiere cajas por grano; su entrenamiento sigue pendiente de anotaciones.


In [ ]:
from pathlib import Path
import csv
from collections import Counter
import subprocess
import sys
import torch

ROOT = Path.cwd().resolve()
while not (ROOT / 'src').is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / 'src').is_dir():
    raise FileNotFoundError('No encuentro la raíz del proyecto (carpeta src).')
INDEX = ROOT / 'data' / 'splits' / 'index.csv'
if not INDEX.is_file():
    raise FileNotFoundError('Falta data/splits/index.csv. Ejecuta primero 02_exploracion_dataset.ipynb.')
with INDEX.open(newline='', encoding='utf-8') as f:
    records = list(csv.DictReader(f))
if not records:
    raise ValueError('El índice está vacío. Regenera las particiones desde el notebook 02.')
foreign = [row['source_path'] for row in records if '/USK-Coffee/train/' not in row['source_path'].replace('\\', '/') ]
if foreign:
    raise ValueError(f'Hay {len(foreign)} imágenes que no provienen de USK-Coffee/train. Regenera las particiones.')
for split in ('train', 'val', 'test'):
    if not (ROOT / 'data' / 'splits' / split).is_dir():
        raise FileNotFoundError(f'Falta la partición {split}.')
print(f'Proyecto: {ROOT}')
print(f'Imágenes registradas desde USK-Coffee/train: {len(records)}')
for split in ('train', 'val', 'test'):
    counts = Counter(row['class'] for row in records if row['split'] == split)
    print(f'{split}: {sum(counts.values())} imágenes {dict(counts)}')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Dispositivo de entrenamiento:', DEVICE)
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'GPU {i}: {torch.cuda.get_device_name(i)}')
else:
    print('AVISO: CUDA no está disponible en este kernel; el entrenamiento irá por CPU.')


## Opción 1: YOLOv8-cls

Clasificador YOLOv8 para imágenes etiquetadas por carpeta. Necesita internet la primera vez para bajar `yolov8n-cls.pt` si esos pesos aún no están en caché. Ajustes en `config/yolo.yaml`; checkpoint en `checkpoints/yolov8_cls/weights/best.pt`.


In [ ]:
subprocess.run([sys.executable, '-m', 'src.train', '--model', 'yolov8_cls'], cwd=ROOT, check=True)


## Opción 2: Slim-CNN

CNN compacta construida con bloques convolucionales separables. Su configuración (`config/slim_cnn.yaml`) define máximo 100 épocas, paciencia de early stop 15 y `min_delta=0.0001`, monitoreando `val_loss`. Con 1,920 imágenes y batch 32 se esperan 60 iteraciones de optimización por época (hasta 6,000 si alcanza el máximo). La consola informa el avance cada 10 iteraciones.

Una corrida nueva conserva sus propios archivos. Si se interrumpe, vuelve a ejecutar esta celda con `RESUME_SLIM = True`; se reanuda desde el último epoch completado. Para otra corrida independiente, cambia `SLIM_OUTPUT` a otra carpeta.


In [ ]:
SLIM_OUTPUT = ROOT / 'checkpoints' / 'slim_cnn'
RESUME_SLIM = False  # Cambia a True solo si esa misma corrida se interrumpió.
command = [sys.executable, '-m', 'src.train', '--model', 'slim_cnn', '--output', str(SLIM_OUTPUT)]
if RESUME_SLIM:
    command.append('--resume')
print('Salida:', SLIM_OUTPUT, '| reanudar:', RESUME_SLIM)
subprocess.run(command, cwd=ROOT, check=True)


## Opción 3: ResNet18 siamesa

Dos ramas comparten los pesos de ResNet18 y aprenden con pares positivos y negativos. La celda crea los manifiestos de pares y entrena con máximo 100 épocas y parada temprana por `val_loss`. Los resultados incluyen `best.pt`, `last.pt` e historiales en `checkpoints/siamese_resnet18/`.


In [ ]:
subprocess.run([sys.executable, '-m', 'src.data.generate_pairs'], cwd=ROOT, check=True)
subprocess.run([sys.executable, '-m', 'src.train', '--model', 'siamese_resnet18'], cwd=ROOT, check=True)


## Detección YOLOv8 para fotos con varios granos (pendiente de anotación)

El entrenamiento de detección tiene máximo 100 épocas y paciencia de 15, pero todavía no se puede iniciar: faltan etiquetas por caja para cada grano. Se esperan `data/detection/images/{train,val,test}` y `data/detection/labels/{train,val,test}`; cada TXT debe tener una línea por objeto: `class_id centro_x centro_y ancho alto`, coordenadas normalizadas entre 0 y 1; IDs `0=sano`, `1=defecto`. Las escenas de una misma hoja o sesión deben quedar en una sola partición.

Cuando los datos estén anotados y revisados, ejecuta la celda de entrenamiento de detección. Ultralytics guardará sus checkpoints `best.pt` y `last.pt` en una carpeta de corrida propia.


Los entrenamientos seleccionan el mejor checkpoint usando validación y nunca usan test para elegir época. Slim-CNN y la siamesa permiten reanudar con `python -m src.train --model slim_cnn --resume` o `python -m src.train --model siamese_resnet18 --resume`, manteniendo la ruta `--output` original. Para YOLO, pasa el `last.pt` mediante `--resume ruta/al/last.pt`. Al terminar, evalúa una vez en test con `python -m src.evaluate_all --model slim_cnn` (o `yolov8_cls` / `siamese_resnet18`).
